#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "semiestrcturprojectdev")
dbutils.widgets.text("catalogo", "project_sedev")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_inicio = f"{catalogo}.bronze"
path_target = f"{catalogo}.silver"

In [0]:
from pyspark.sql.functions import from_json, col, explode, when, size, array_min, array_max
from pyspark.sql.types import ArrayType, DoubleType

##leer tabla bronze

In [0]:
### leer el json y explorar las claves valor
df_i = spark.table( f"{catalogo}.bronze.instances_train") 
df_i.printSchema()

# EXTRAER ANNOTATIONS

In [0]:
ann = df_i.select(explode("annotations").alias("annot")) \
                .select("annot.*")

In [0]:
### ver cuantas columnas tiene bbox dentro de annotations
#ann.select("bbox").show(5, truncate=False)

In [0]:
### flatear las columnas del array
ann_flt = ann.selectExpr(
    "*",
    "bbox[0] as bbox_x",
    "bbox[1] as bbox_y",
    "bbox[2] as bbox_width",
    "bbox[3] as bbox_height"
).drop("bbox")

#ann_flt.limit(5).display()

### validacion segmentacion string en annotations
pasar la columna a double array para ver si tiene valores corruptos

In [0]:
### como el array es un tring, toca convertirlo a array de double
schema = ArrayType(ArrayType(DoubleType()))

ann_flt = ann_flt.withColumn(
    "segmentation_parsed",
    from_json(col("segmentation"), schema)
)

In [0]:
### filtrar los que si pasaron y los que no
corruptos = ann_flt.filter(col("segmentation_parsed").isNull())
validos = ann_flt.filter(col("segmentation_parsed").isNotNull())

#validos.count()

In [0]:
### al visualizar los corruptos nos damos de cuenta que no es un json corrpto si no un RLE (run-length-encoding)
#corruptos.limit(3).display()

### validacion segunda del array segmentacion dentro de annotations

ya que el array tiene dos tipos toca trabajarlo de otra forma

In [0]:
### vsi contiene la palabra count es RLE y si no es json

ann_flt_val = (
    ann_flt.drop("segmentation_parsed")
    .withColumn(
        "segmentation_type",
        when(col("segmentation").contains("counts"), "RLE")
        .when(col("segmentation").contains("[["), "polygon")
        .otherwise("unknown")
        )
)

### validamos que solo haya dos tipos
#ann_flt_val.select('segmentation_type').distinct().show()

In [0]:
### separar dataframes

rle_df = ann_flt_val.filter(col("segmentation_type") == "RLE")
polygon_df = ann_flt_val.filter(col("segmentation_type") == "polygon")

In [0]:
### guardar los rle en un dataframe
rle_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{path_target}.annotations_rle_raw")

### convertir de nuevo a array

In [0]:
### como el array es un tring, toca convertirlo a array de double
schema = ArrayType(ArrayType(DoubleType()))

polygon_df = polygon_df.withColumn(
    "segmentation_parsed",
    from_json(col("segmentation"), schema)
)

In [0]:
### VISUALIZAR CUANTOS PUNTOS DISTINTOS TIENE LA COLUMNA
#polygon_df.select(size("segmentation_parsed")).distinct().show()

In [0]:
polygon_df = polygon_df \
    .withColumn("num_points", size("segmentation_parsed")) \
    .withColumn("min_val", array_min("segmentation_parsed")) \
    .withColumn("max_val", array_max("segmentation_parsed"))

In [0]:
## guardar los limpios wn la tabla clean
polygon_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{path_target}.annotations_polygon_clean")

# EXTRAER IMAGES

In [0]:

images = df_i.select(explode("images").alias("image")).select("image.*")

categories = df_i.select(explode("categories").alias("category")) \
               .select("category.*")

images.display()